## Exercise 1: Set Up the Catalog Structure

You are a data engineer at **Lakeside University**, a mid-sized institution digitizing its academic operations. Your team is building a data platform in Azure Databricks and Unity Catalog to manage student records, course catalogues, and enrollment data.

In this exercise, you create the top-level namespace structure: one `catalog` and three `schemas` following the **medallion architecture** naming pattern.

Here's the full Unity Catalog structure you will create:

```txt

Unity Catalog
└── edu_dev  [catalog]
    │   tags: environment=development | university=lakeside | data_classification=internal
    │
    ├── bronze  [schema]
    │   └── raw_files/  [managed volume]
    │       └── students.csv
    │
    ├── silver  [schema]
    │   ├── students  [table]
    │   │   ├── student_id       BIGINT  NOT NULL  ← PK
    │   │   ├── first_name       STRING
    │   │   ├── last_name        STRING
    │   │   ├── email            STRING
    │   │   ├── enrollment_year  INT
    │   │   ├── program          STRING
    │   │   └── phone_number     STRING  (added in Exercise 6)
    │   │
    │   ├── courses  [table]
    │   │   ├── course_id    BIGINT  NOT NULL  ← PK
    │   │   ├── course_name  STRING
    │   │   ├── department   STRING
    │   │   └── credits      INT
    │   │
    │   ├── enrollments  [table]
    │   │   ├── enrollment_id  BIGINT  NOT NULL  ← PK
    │   │   ├── student_id     BIGINT            → FK → students.student_id
    │   │   ├── course_id      BIGINT            → FK → courses.course_id
    │   │   ├── semester       STRING
    │   │   └── grade          DECIMAL(4,2)
    │   │
    │   ├── vw_student_enrollments  [view]
    │   │   └── joins students + courses + enrollments
    │   │
    │   └── get_grade_classification(grade)  [function]
    │       └── returns STRING  (A / B / C / D / F)
    │
    └── gold  [schema]
        └── vw_department_enrollment_stats  [materialized view]
            └── aggregates enrollments + courses
                (department, total_enrollments, avg_grade, distinct_students)
```

### Task 1.1 — Create the development catalog

Create a catalog called `edu_dev` with a descriptive comment. This catalog serves as the **development environment** for Lakeside University's data platform.

**Naming convention reminders:**
- Use lowercase with underscores
- No periods, spaces, forward slashes, or control characters

> 🤖 **Databricks Assistant tip:** Ask the Assistant:
> *"How do I create a Unity Catalog catalog in Databricks SQL with a comment?"*

**Hint:** Use `CREATE CATALOG IF NOT EXISTS` followed by a `COMMENT` string.

In [ ]:
# Determine the storage root
storage_root = (
    spark.sql("DESCRIBE CATALOG EXTENDED adb_dp750")
    .filter("info_name = 'Storage Root'")
    .select("info_value")
    .first()[0]
)
print(f"Storage root: {storage_root}")

spark.sql(f"""
    CREATE CATALOG IF NOT EXISTS edu_dev
    MANAGED LOCATION '{storage_root}'
    COMMENT 'Development environment for Lakeside University data platform'
""")

### Task 1.2 — Create medallion schemas

Within the `edu_dev` catalog, create three schemas that implement the **medallion architecture**:

| Schema | Purpose |
|--------|---------|
| `bronze` | Raw ingested data — unmodified source files |
| `silver` | Cleaned, validated, and enriched data |
| `gold` | Aggregated, analytics-ready data |

> 🤖 **Databricks Assistant tip:** Ask:
> *"Show me how to create multiple schemas in a Databricks Unity Catalog catalog using SQL."*

**Hint:** Use `CREATE SCHEMA IF NOT EXISTS catalog.schema_name` for each schema.

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS edu_dev.bronze
COMMENT 'Raw ingested data — unmodified source files';

CREATE SCHEMA IF NOT EXISTS edu_dev.silver
COMMENT 'Cleaned, validated, and enriched data';

CREATE SCHEMA IF NOT EXISTS edu_dev.gold
COMMENT 'Aggregated, analytics-ready data';

## Exercise 2: Create Tables with Constraints

With the catalog structure in place, create the core tables for Lakeside University's academic data in the `silver` schema. You'll define **primary key** and **foreign key** constraints to document the data model. Note that in Unity Catalog, these constraints are informational — they help optimize queries but do not enforce referential integrity at write time.

### Task 2.1 — Create the `students` table

Create a managed Delta table `edu_dev.silver.students` to store student records with the following columns:

| Column | Type | Description |
|--------|------|-------------|
| `student_id` | BIGINT | Unique student identifier (primary key) |
| `first_name` | STRING | Student's first name |
| `last_name` | STRING | Student's last name |
| `email` | STRING | University email address |
| `enrollment_year` | INT | Year the student enrolled |
| `program` | STRING | Degree program |

> 🤖 **Databricks Assistant tip:** Ask:
> *"How do I create a managed Delta table with a primary key constraint in Unity Catalog?"*

**Hint:** Add `CONSTRAINT students_pk PRIMARY KEY (student_id)` at the end of the column list.

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS edu_dev.silver.students (
  student_id      BIGINT NOT NULL,
  first_name      STRING,
  last_name       STRING,
  email           STRING,
  enrollment_year INT,
  program         STRING,
  CONSTRAINT students_pk PRIMARY KEY (student_id)
);

### Task 2.2 — Create the `courses` table

Create `edu_dev.silver.courses` with the following columns:

| Column | Type | Description |
|--------|------|-------------|
| `course_id` | BIGINT | Unique course identifier (primary key) |
| `course_name` | STRING | Full name of the course |
| `department` | STRING | Academic department offering the course |
| `credits` | INT | Number of academic credits |

> 🤖 **Databricks Assistant tip:** Ask:
> *"Write a CREATE TABLE statement for a courses table in Databricks Unity Catalog with a primary key constraint."*

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS edu_dev.silver.courses (
  course_id   BIGINT NOT NULL,
  course_name STRING,
  department  STRING,
  credits     INT,
  CONSTRAINT courses_pk PRIMARY KEY (course_id)
);

### Task 2.3 — Create the `enrollments` table with foreign keys

Create `edu_dev.silver.enrollments` to record which students are enrolled in which courses. This table must reference both `students` and `courses` via foreign key constraints.

| Column | Type | Description |
|--------|------|-------------|
| `enrollment_id` | BIGINT | Unique enrollment record (primary key) |
| `student_id` | BIGINT | References `students.student_id` |
| `course_id` | BIGINT | References `courses.course_id` |
| `semester` | STRING | Academic semester (e.g., `Spring 2023`) |
| `grade` | DECIMAL(4,2) | Numerical grade on a 0.0–10.0 scale |

> 🤖 **Databricks Assistant tip:** Ask:
> *"How do I add foreign key constraints referencing other tables when creating a table in Databricks Unity Catalog?"*

**Hint:** Use `CONSTRAINT <name> FOREIGN KEY (<col>) REFERENCES <catalog>.<schema>.<table>(<col>)` for each foreign key.

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS edu_dev.silver.enrollments (
  enrollment_id BIGINT NOT NULL,
  student_id    BIGINT,
  course_id     BIGINT,
  semester      STRING,
  grade         DECIMAL(4,2),
  CONSTRAINT enrollments_pk         PRIMARY KEY (enrollment_id),
  CONSTRAINT enrollments_students_fk FOREIGN KEY (student_id) REFERENCES edu_dev.silver.students(student_id),
  CONSTRAINT enrollments_courses_fk  FOREIGN KEY (course_id)  REFERENCES edu_dev.silver.courses(course_id)
);

### Sample data — run this cell

The following cell inserts sample data into your three tables. **Run it without modification** before continuing to Exercise 3.

In [ ]:
%sql
-- Populate students
INSERT INTO edu_dev.silver.students VALUES
(1001, 'Emma',   'Watson',    'emma.watson@lakeside.edu',    2022, 'Computer Science'),
(1002, 'Liam',   'Ahmed',     'liam.ahmed@lakeside.edu',     2023, 'Data Science'),
(1003, 'Sofia',  'Chen',      'sofia.chen@lakeside.edu',     2021, 'Mathematics'),
(1004, 'James',  'Okonkwo',   'james.okonkwo@lakeside.edu',  2022, 'Computer Science'),
(1005, 'Aisha',  'Patel',     'aisha.patel@lakeside.edu',    2023, 'Data Science'),
(1006, 'Noah',   'Yamamoto',  'noah.yamamoto@lakeside.edu',  2020, 'Mathematics'),
(1007, 'Olivia', 'Kowalski',  'olivia.kowalski@lakeside.edu',2021, 'Computer Science'),
(1008, 'Carlos', 'Reyes',     'carlos.reyes@lakeside.edu',   2022, 'Data Science'),
(1009, 'Mei',    'Lindqvist', 'mei.lindqvist@lakeside.edu',  2023, 'Mathematics'),
(1010, 'Jake',   'OBrien',    'jake.obrien@lakeside.edu',    2020, 'Computer Science');

-- Populate courses
INSERT INTO edu_dev.silver.courses VALUES
(101, 'Introduction to Programming',   'Computer Science', 3),
(102, 'Data Structures',               'Computer Science', 4),
(103, 'Statistics for Data Science',   'Data Science',     3),
(104, 'Machine Learning Fundamentals', 'Data Science',     4),
(105, 'Calculus I',                    'Mathematics',      4),
(106, 'Linear Algebra',                'Mathematics',      3);

-- Populate enrollments
INSERT INTO edu_dev.silver.enrollments VALUES
(1,  1001, 101, 'Spring 2023', 8.50),
(2,  1001, 102, 'Fall 2023',   7.80),
(3,  1002, 103, 'Spring 2023', 9.20),
(4,  1002, 104, 'Fall 2023',   6.50),
(5,  1003, 105, 'Spring 2022', 7.00),
(6,  1003, 106, 'Fall 2022',   8.00),
(7,  1004, 101, 'Fall 2022',   6.80),
(8,  1005, 103, 'Spring 2023', 9.00),
(9,  1006, 105, 'Fall 2020',   7.50),
(10, 1007, 102, 'Spring 2022', 8.20),
(11, 1008, 104, 'Fall 2023',   7.30),
(12, 1009, 106, 'Spring 2023', 8.80),
(13, 1010, 101, 'Fall 2021',   5.50),
(14, 1004, 102, 'Spring 2023', 7.10),
(15, 1005, 104, 'Fall 2023',   8.90);

## Exercise 3: Create Views

Views provide a virtual layer over your tables, simplifying access to joined data and pre-computing expensive aggregations. In this exercise, you create a **standard view** for combined enrollment records and a **materialized view** in the `gold` schema to serve the Lakeside University analytics team with pre-aggregated department statistics.

### Task 3.1 — Create a standard view

Create a view `edu_dev.silver.vw_student_enrollments` that presents a combined, human-readable view of student enrollment records. It should include:

- Student full name (concatenate `first_name` and `last_name` with a space between them)
- Student `email`
- `course_name`
- `department`
- `semester`
- `grade`

> 🤖 **Databricks Assistant tip:** Ask:
> *"Write a SQL view in Azure Databricks that joins three tables: students, courses, and enrollments"*

**Hint:** Use `CONCAT(first_name, ' ', last_name)` or the `||` operator to combine names. You need to join all three silver tables.

In [ ]:
%sql
CREATE OR REPLACE VIEW edu_dev.silver.vw_student_enrollments AS
SELECT
  CONCAT(s.first_name, ' ', s.last_name) AS full_name,
  s.email,
  c.course_name,
  c.department,
  e.semester,
  e.grade
FROM edu_dev.silver.enrollments e
JOIN edu_dev.silver.students s ON e.student_id = s.student_id
JOIN edu_dev.silver.courses  c ON e.course_id  = c.course_id;

### Task 3.2 — Create a materialized view for department statistics

Create a materialized view `edu_dev.gold.vw_department_enrollment_stats` that pre-computes the following per academic department:

| Output column | Description |
|--------------|-------------|
| `department` | Academic department name |
| `total_enrollments` | Total number of enrollment records |
| `avg_grade` | Average grade, rounded to 2 decimal places |
| `distinct_students` | Number of unique students enrolled |

> 🤖 **Databricks Assistant tip:** Ask:
> *"What is the syntax for creating a materialized view in Azure Databricks that aggregates data from two joined tables?"*

**Hint:** Use `CREATE MATERIALIZED VIEW`, join `enrollments` with `courses`, and use `COUNT()`, `ROUND(AVG(...), 2)`, and `COUNT(DISTINCT ...)`.

> [!NOTE]
> Materialized views require a SQL warehouse to run. If this cell fails, check the compute selector at the top of the notebook and switch from **Serverless** compute to a **Serverless SQL warehouse**.

In [ ]:
%sql
CREATE MATERIALIZED VIEW IF NOT EXISTS edu_dev.gold.vw_department_enrollment_stats AS
SELECT
  c.department,
  COUNT(*)                     AS total_enrollments,
  ROUND(AVG(e.grade), 2)       AS avg_grade,
  COUNT(DISTINCT e.student_id) AS distinct_students
FROM edu_dev.silver.enrollments e
JOIN edu_dev.silver.courses c ON e.course_id = c.course_id
GROUP BY c.department;

**✅ Test your materialized view** — Run the following query to verify it returns per-department aggregates:

You should see one row per academic department with columns for `total_enrollments`, `avg_grade`, and `distinct_students`.

In [ ]:
%sql
SELECT * FROM edu_dev.gold.vw_department_enrollment_stats;

## Exercise 4: Create a Volume and Read Files

Volumes extend Unity Catalog governance to files, not just structured tables. In this exercise, you create a **managed volume** in the `bronze` schema to serve as a landing zone for raw CSV files — simulating files arriving from Lakeside University's external student information system.

### Task 4.1 — Create a managed volume

Create a managed volume named `raw_files` in the `edu_dev.bronze` schema. This will serve as the landing area for raw data files.

> 🤖 **Databricks Assistant tip:** Ask:
> *"How do I create a managed volume in Unity Catalog using a SQL statement?"*

**Hint:** Use `CREATE VOLUME IF NOT EXISTS <catalog>.<schema>.<volume>`.

In [ ]:
%sql
CREATE VOLUME IF NOT EXISTS edu_dev.bronze.raw_files
COMMENT 'Landing area for raw CSV files from the student information system';

### Task 4.2 — Write a CSV file to the volume

The following Python cell writes a sample student CSV file directly into your new volume. **Run it without modification.**

In [ ]:
# Write a sample student CSV file to the volume — run this cell without modification
csv_content = """student_id,first_name,last_name,email,enrollment_year,program
1001,Emma,Watson,emma.watson@lakeside.edu,2022,Computer Science
1002,Liam,Ahmed,liam.ahmed@lakeside.edu,2023,Data Science
1003,Sofia,Chen,sofia.chen@lakeside.edu,2021,Mathematics
1004,James,Okonkwo,james.okonkwo@lakeside.edu,2022,Computer Science
1005,Aisha,Patel,aisha.patel@lakeside.edu,2023,Data Science
1006,Noah,Yamamoto,noah.yamamoto@lakeside.edu,2020,Mathematics
1007,Olivia,Kowalski,olivia.kowalski@lakeside.edu,2021,Computer Science
1008,Carlos,Reyes,carlos.reyes@lakeside.edu,2022,Data Science
1009,Mei,Lindqvist,mei.lindqvist@lakeside.edu,2023,Mathematics
1010,Jake,OBrien,jake.obrien@lakeside.edu,2020,Computer Science"""

dbutils.fs.put("/Volumes/edu_dev/bronze/raw_files/students.csv", csv_content, overwrite=True)
print("students.csv written to /Volumes/edu_dev/bronze/raw_files/")

### Task 4.3 — Read the CSV from the volume into a DataFrame

Read the CSV file you just placed in the volume into a Spark DataFrame and display its contents. The file is at `/Volumes/edu_dev/bronze/raw_files/students.csv`.

> 🤖 **Databricks Assistant tip:** Ask:
> *"How do I read a CSV file with a header row from a Unity Catalog volume path in PySpark?"*

**Hint:** Use `spark.read.csv(path, header=True, inferSchema=True)` then call `.display()` or `.show()` on the result.

In [ ]:
df = spark.read.csv(
    "/Volumes/edu_dev/bronze/raw_files/students.csv",
    header=True,
    inferSchema=True
)
display(df)

## Exercise 5: Create a Reusable SQL Function

Unity Catalog allows you to create **governed, reusable functions** that encapsulate business logic. Instead of repeating grade conversion logic in every query, you'll write it once as a SQL function and call it from any notebook or query in the workspace.

### Task 5.1 — Create a grade classification function

Create a SQL scalar function `edu_dev.silver.get_grade_classification` that accepts a `grade DECIMAL(4,2)` and returns a `STRING` letter classification based on the scale used at Lakeside University:

| Grade range | Classification |
|-------------|----------------|
| ≥ 8.5 | `'A'` |
| ≥ 7.0 | `'B'` |
| ≥ 5.5 | `'C'` |
| ≥ 4.0 | `'D'` |
| < 4.0 | `'F'` |

> 🤖 **Databricks Assistant tip:** Ask:
> *"How do I create a SQL scalar function in Unity Catalog that uses a CASE expression to return a letter grade based on a numeric score?"*

**Hint:** Use `CREATE FUNCTION ... RETURNS STRING RETURN CASE WHEN ... END`.

In [ ]:
%sql
CREATE OR REPLACE FUNCTION edu_dev.silver.get_grade_classification(grade DECIMAL(4,2))
RETURNS STRING
COMMENT 'Returns a letter grade classification based on the Lakeside University grading scale'
RETURN CASE
  WHEN grade >= 8.5 THEN 'A'
  WHEN grade >= 7.0 THEN 'B'
  WHEN grade >= 5.5 THEN 'C'
  WHEN grade >= 4.0 THEN 'D'
  ELSE 'F'
END;

### Task 5.2 — Test the function

Query `edu_dev.silver.enrollments` and apply your new function to produce a `grade_classification` column for each row. Include: `enrollment_id`, `student_id`, `course_id`, `semester`, `grade`, and `grade_classification`.

> 🤖 **Databricks Assistant tip:** Ask:
> *"How do I call a Unity Catalog user-defined function in a SELECT statement in Azure Databricks?"*

In [ ]:
%sql
SELECT
  enrollment_id,
  student_id,
  course_id,
  semester,
  grade,
  edu_dev.silver.get_grade_classification(grade) AS grade_classification
FROM edu_dev.silver.enrollments
ORDER BY enrollment_id;

## Exercise 6: DDL Operations

Unity Catalog objects evolve over time as requirements change. In this exercise, you use `ALTER` statements to:
1. Extend the `students` table with a new column.
2. Apply governance **tags** to the `edu_dev` catalog.

### Task 6.1 — Add a column to the students table

The student registration system has been updated to capture phone numbers. Add a nullable `phone_number` column of type `STRING` to `edu_dev.silver.students`.

> 🤖 **Databricks Assistant tip:** Ask:
> *"How do I add a new nullable column to an existing Delta table in Azure Databricks using SQL?"*

**Hint:** Use `ALTER TABLE`.

In [ ]:
%sql
ALTER TABLE edu_dev.silver.students
ADD COLUMN phone_number STRING
COMMENT 'Student contact phone number';

### Task 6.2 — Apply tags to the catalog

Apply the following tags to the `edu_dev` catalog to support governance and discoverability:

| Tag key | Tag value |
|---------|----------|
| `environment` | `'development'` |
| `university` | `'lakeside'` |
| `data_classification` | `'internal'` |

> 🤖 **Databricks Assistant tip:** Ask:
> *"How do I set metadata tags on a Unity Catalog catalog using the ALTER CATALOG statement in Databricks?"*

In [ ]:
%sql
ALTER CATALOG edu_dev SET TAGS (
  'environment'         = 'development',
  'university'          = 'lakeside',
  'data_classification' = 'internal'
);

### Task 6.3 — Verify your work

Run the following cells to confirm your catalog structure, table schemas, and tags are all in place. Compare the output with what you created throughout this lab.

In [ ]:
%sql
-- Show all tables and views in the silver schema
SHOW TABLES IN edu_dev.silver;

In [ ]:
%sql
-- Describe the students table — verify the new phone_number column is present
DESCRIBE TABLE edu_dev.silver.students;

In [ ]:
%sql
-- Show catalog details
DESCRIBE CATALOG EXTENDED edu_dev;

In [ ]:
%sql 
-- Query tags via information schema (structured way)
SELECT *
FROM system.information_schema.catalog_tags
WHERE catalog_name = 'edu_dev';

---

## Next steps

You have completed all notebook exercises. Return to the **lab instructions** and continue with:

- **Exercise**: Configure an AI/BI Genie Space (optional)